# Importing libraries

In [3]:
# Basic libraries
import pandas as pd
from datasets import load_dataset
import time
import pickle


# Classification models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB

# Vectorizers
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Utilities and metrics
from itertools import product
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from memory_profiler import memory_usage

# Preprocessing
import nltk
import re

# Download nltk resources
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rafael\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Setting seeds

In [4]:
s1 = 2
s2 = 3
s3 = 5

seeds = [s1, s2, s3]

# Importing datasets

In [5]:
ds = load_dataset("KushT/bbc_news_multiclass_train_val_test")

train = ds['train'].to_pandas()
val = ds['validation'].to_pandas()
test = ds['test'].to_pandas()

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1512 entries, 0 to 1511
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1512 non-null   object
 1   label   1512 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 23.8+ KB


# Dataset preprocessing

In [6]:
stop_words = set(nltk.corpus.stopwords.words('english'))
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()

    text = re.sub(r'[^\w\s]', '', text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(words)
train['text'] = train['text'].apply(preprocess_text)

# GridSearch implementation

In [7]:
vectorizers = [
    TfidfVectorizer(),
    CountVectorizer()
]

models = {
    'RandomForest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'criterion': ['gini', 'entropy', 'log_loss']
        }
    },
    'SVC': {
        'model': SVC(),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf', 'sigmoid']
        }
    },
    'MultinomialNB': {
        'model': MultinomialNB(),
        'params': {
            'alpha': [0.01, 0.1, 1.0]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=1000),
        'params': {
            'C': [0.1, 1, 10],
            'penalty': ['l2']
        }
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7],
            'algorithm': ['ball_tree', 'kd_tree', 'brute']
        }
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(),
        'params': {
            'n_estimators': [100, 150, 200],
            'criterion': ['friedman_mse', 'squared_error'],
        }
    },
    'AdaBoost': {
        'model': AdaBoostClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'learning_rate': [0.01, 0.1, 1.0]
        }
    },
    'SGD': {
        'model': SGDClassifier(),
        'params': {
            'alpha': [0.0001, 0.001, 0.01],
            'penalty': ['l2', 'l1', 'elasticnet']
        }
    }
}

In [8]:
columns = ['seed', 'vectorizer', 'model', 'params', 'accuracy', 'training_time', 'prediction_time', 'peak_memory_train', 'peak_memory_prediction']

classes = sorted(train['label'].unique())
for c in classes:
    columns.extend([
        f'precision_class_{c}',
        f'recall_class_{c}',
        f'f1_class_{c}'
    ])

results = pd.DataFrame(columns=columns)

In [9]:
for seed in seeds:
    print(f"Processing seed: {seed}")
    for vectorizer in vectorizers:
        print(f"Processing vectorizer: {vectorizer.__class__.__name__}")
        for name, info in models.items():
            print(f"Processing model: {name}")

            model = info['model']
            param_grid = info['params']
            param_combinations = product(*param_grid.values())
            
            for combination in param_combinations:
                params = dict(zip(param_grid.keys(), combination))
                model.set_params(**params)
                if 'random_state' in model.get_params():
                    model.set_params(random_state=seed)

                print(f"Training {name} with params {params} and vectorizer {vectorizer.__class__.__name__}")

                pipeline = Pipeline([
                    ('vectorizer', vectorizer),
                    ('model', model)
                ])

                def train_model():
                    pipeline.fit(train['text'], train['label'])

                def predict_model():
                    return pipeline.predict(val['text'])

                # Training Phase
                start_time = time.perf_counter()
                peak_memory_train = memory_usage(train_model, max_usage=True)
                train_time = time.perf_counter() - start_time
                print(f"Training time: {train_time}")
                print(f"Peak memory usage during training: {peak_memory_train} MB")

                # Prediction Phase
                start_time = time.perf_counter()
                peak_memory_pred, y_pred = memory_usage(predict_model, max_usage=True, retval=True)
                prediction_time = time.perf_counter() - start_time
                print(f"Prediction time: {prediction_time}")
                print(f"Peak memory usage during prediction: {peak_memory_pred} MB")
                
                accuracy = accuracy_score(val['label'], y_pred)
                precisions, recalls, f1s, supports = precision_recall_fscore_support(val['label'], y_pred, average=None, labels=classes, zero_division=0)

                result_dict = {
                    'seed': seed,
                    'vectorizer': vectorizer.__class__.__name__,
                    'model': name,
                    'params': params,
                    'accuracy': accuracy,
                    'training_time': train_time,
                    'prediction_time': prediction_time,
                    'peak_memory_train': peak_memory_train,
                    'peak_memory_prediction': peak_memory_pred,
                }

                for i, c in enumerate(classes):
                    result_dict[f'precision_class_{c}'] = precisions[i]
                    result_dict[f'recall_class_{c}'] = recalls[i]
                    result_dict[f'f1_class_{c}'] = f1s[i]

                result = pd.DataFrame([result_dict])
                results = pd.concat([results, result], ignore_index=True)
                print("-"*100)

results.to_csv('results/results_sklearn_multiclass2.csv', index=False)

Processing seed: 2
Processing vectorizer: TfidfVectorizer
Processing model: RandomForest
Training RandomForest with params {'n_estimators': 50, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 1.106778800007305
Peak memory usage during training: 384.79296875 MB
Prediction time: 1.0681335999979638
Peak memory usage during prediction: 384.8515625 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'entropy'} and vectorizer TfidfVectorizer


C:\Users\Rafael\AppData\Local\Temp\ipykernel_15560\3414761210.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, result], ignore_index=True)


Training time: 1.1908806999999797
Peak memory usage during training: 387.8828125 MB
Prediction time: 1.0680489999940619
Peak memory usage during prediction: 385.8125 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'log_loss'} and vectorizer TfidfVectorizer
Training time: 1.1871726999961538
Peak memory usage during training: 388.69921875 MB
Prediction time: 1.0640425000019604
Peak memory usage during prediction: 385.8515625 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 100, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 1.563360999993165
Peak memory usage during training: 388.14453125 MB
Prediction time: 1.0795954999921378
Peak memory usage during prediction: 388.84765625 MB
----------------------------------------------------------------

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")


Training time: 1.276418100009323
Peak memory usage during training: 406.8359375 MB
Prediction time: 1.1055532000027597
Peak memory usage during prediction: 409.85546875 MB
----------------------------------------------------------------------------------------------------
Training KNeighbors with params {'n_neighbors': 3, 'algorithm': 'kd_tree'} and vectorizer TfidfVectorizer


...

Training time: 3.4020289000036428
Peak memory usage during training: 407.3984375 MB
Prediction time: 1.1289216000004672
Peak memory usage during prediction: 398.25 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'n_estimators': 150, 'learning_rate': 0.1} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Training time: 3.4281303999887314
Peak memory usage during training: 406.40234375 MB
Prediction time: 1.131442199999583
Peak memory usage during prediction: 398.3046875 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'n_estimators': 150, 'learning_rate': 1.0} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Training time: 3.396874800004298
Peak memory usage during training: 407.11328125 MB
Prediction time: 1.1290780999988783
Peak memory usage during prediction: 398.1171875 MB
----------------------------------------------------------------------------------------------------
Processing model: SGD
Training SGD with params {'alpha': 0.0001, 'penalty': 'l2'} and vectorizer CountVectorizer
Training time: 1.3380876000010176
Peak memory usage during training: 406.66796875 MB
Prediction time: 1.048709399998188
Peak memory usage during prediction: 399.36328125 MB
----------------------------------------------------------------------------------------------------
Training SGD with params {'alpha': 0.0001, 'penalty': 'l1'} and vectorizer CountVectorizer
Training time: 0.7540000999870244
Peak memory usage during training: 407.2265625 MB
Prediction time: 1.0491976000048453
Peak memory usage during prediction: 402.828125 MB
------------------------------------------------------------------------------

# Process results

In [10]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   seed                    396 non-null    object 
 1   vectorizer              396 non-null    object 
 2   model                   396 non-null    object 
 3   params                  396 non-null    object 
 4   accuracy                396 non-null    float64
 5   training_time           396 non-null    float64
 6   prediction_time         396 non-null    float64
 7   peak_memory_train       396 non-null    float64
 8   peak_memory_prediction  396 non-null    float64
 9   precision_class_0       396 non-null    float64
 10  recall_class_0          396 non-null    float64
 11  f1_class_0              396 non-null    float64
 12  precision_class_1       396 non-null    float64
 13  recall_class_1          396 non-null    float64
 14  f1_class_1              396 non-null    fl

In [11]:
results.head()

,seed,vectorizer,model,params,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_0,...,f1_class_1,precision_class_2,recall_class_2,f1_class_2,precision_class_3,recall_class_3,f1_class_3,precision_class_4,recall_class_4,f1_class_4
0,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'gini'}",0.931398,1.106779,1.068134,384.792969,384.851562,0.870000,...,0.907692,0.955882,0.915493,0.935252,0.945652,1.0,0.972067,1.000000,0.808824,0.894309
1,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'entropy'}",0.920844,1.190881,1.068049,387.882812,385.812500,0.913978,...,0.907692,0.984615,0.901408,0.941176,0.870000,1.0,0.930481,0.947368,0.794118,0.864000
2,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'log_loss'}",0.920844,1.187173,1.064043,388.699219,385.851562,0.913978,...,0.907692,0.984615,0.901408,0.941176,0.870000,1.0,0.930481,0.947368,0.794118,0.864000
3,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'gini'}",0.941953,1.563361,1.079595,388.144531,388.847656,0.905263,...,0.932331,0.969697,0.901408,0.934307,0.935484,1.0,0.966667,1.000000,0.852941,0.920635
4,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'entropy'}",0.934037,1.751320,1.072379,390.558594,388.839844,0.934066,...,0.916031,0.955882,0.915493,0.935252,0.896907,1.0,0.945652,0.982759,0.838235,0.904762


In [12]:
results['params'] = results['params'].astype(str)
results_avg_seed = results.groupby(['model', 'vectorizer', 'params']).mean().reset_index()
results_avg_seed['f1_avg'] = results_avg_seed[[col for col in results_avg_seed.columns if 'f1_class' in col]].mean(axis=1)
results_avg_seed

,model,vectorizer,params,seed,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_0,...,precision_class_2,recall_class_2,f1_class_2,precision_class_3,recall_class_3,f1_class_3,precision_class_4,recall_class_4,f1_class_4,f1_avg
0,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 0.01}",3.333333,0.641161,2.562972,1.139214,409.196615,402.781250,0.493243,...,1.000000,0.450704,0.621359,0.615942,0.977011,0.755556,0.923077,0.352941,0.510638,0.616617
1,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 0.1}",3.333333,0.791557,2.532792,1.116509,408.796875,401.311198,0.666667,...,0.750000,0.887324,0.812903,0.851485,0.988506,0.914894,1.000000,0.352941,0.521739,0.769002
2,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 1.0}",3.333333,0.757256,2.637893,1.135861,409.066406,402.812500,0.707071,...,0.671053,0.718310,0.693878,0.774510,0.908046,0.835979,0.952381,0.588235,0.727273,0.751170
3,AdaBoost,CountVectorizer,"{'n_estimators': 150, 'learning_rate': 0.01}",3.333333,0.672823,3.533606,1.212925,408.891927,399.852865,0.477419,...,1.000000,0.478873,0.647619,0.708333,0.977011,0.821256,0.954545,0.308824,0.466667,0.653282
4,AdaBoost,CountVectorizer,"{'n_estimators': 150, 'learning_rate': 0.1}",3.333333,0.815303,3.547908,1.156837,409.055990,401.335938,0.683333,...,0.777778,0.887324,0.828947,0.924731,0.988506,0.955556,1.000000,0.338235,0.505495,0.788329
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,SVC,TfidfVectorizer,"{'C': 1, 'kernel': 'rbf'}",3.333333,0.965699,3.463356,0.981915,404.225260,402.036458,0.945055,...,0.985294,0.943662,0.964029,0.945652,1.000000,0.972067,1.000000,0.882353,0.937500,0.964993
128,SVC,TfidfVectorizer,"{'C': 1, 'kernel': 'sigmoid'}",3.333333,0.970976,2.212927,0.808624,405.304688,401.169271,0.943820,...,0.971014,0.943662,0.957143,0.988636,1.000000,0.994286,0.984848,0.955882,0.970149,0.970713
129,SVC,TfidfVectorizer,"{'C': 10, 'kernel': 'linear'}",3.333333,0.973615,2.791721,0.817905,405.257812,401.217448,0.954545,...,0.971014,0.943662,0.957143,0.988636,1.000000,0.994286,0.984848,0.955882,0.970149,0.973330
130,SVC,TfidfVectorizer,"{'C': 10, 'kernel': 'rbf'}",3.333333,0.965699,3.447231,0.977326,405.425781,401.761719,0.945055,...,0.985294,0.943662,0.964029,0.945652,1.000000,0.972067,1.000000,0.882353,0.937500,0.964993


In [13]:
best_result = results_avg_seed.loc[results_avg_seed['f1_avg'].idxmax()]

best_model = models[best_result['model']]['model']
best_params = eval(best_result['params'])
best_model.set_params(**best_params)

best_result_vectorizer = eval(best_result['vectorizer'])()

pipeline = Pipeline([
    ('vectorizer', best_result_vectorizer),
    ('model', best_model)
])

pipeline.fit(train['text'], train['label'])
y_pred = pipeline.predict(test['text'])

accuracy = accuracy_score(test['label'], y_pred)
precisions, recalls, f1s, supports = precision_recall_fscore_support(test['label'], y_pred, average=None, labels=classes, zero_division=0)

print(f"Best model: {best_result['model']}")
print(f"Best model params: {best_result['params']}")
print(f"Best vectorizer: {best_result['vectorizer']}")
print(f"Best accuracy: {accuracy}\n")

for i, c in enumerate(classes):
    print(f"Class {c}")
    print(f"Precision: {precisions[i]}")
    print(f"Recall: {recalls[i]}")
    print(f"F1: {f1s[i]}")
    print(f"Support: {supports[i]}\n")

Best model: MultinomialNB
Best model params: {'alpha': 0.1}
Best vectorizer: CountVectorizer
Best accuracy: 0.9850299401197605

Class 0
Precision: 1.0
Recall: 0.9605263157894737
F1: 0.9798657718120806
Support: 76

Class 1
Precision: 0.9824561403508771
Recall: 0.9655172413793104
F1: 0.9739130434782609
Support: 58

Class 2
Precision: 0.984375
Recall: 1.0
F1: 0.9921259842519685
Support: 63

Class 3
Precision: 1.0
Recall: 1.0
F1: 1.0
Support: 77

Class 4
Precision: 0.9523809523809523
Recall: 1.0
F1: 0.975609756097561
Support: 60



In [14]:
with open('models/best_model_sklearn_multiclass2.pkl', 'wb') as f:
    pickle.dump(pipeline, f)